# CNN Retail Image Classification & Confidence Routing

## Problem / objective
Build an end-to-end computer-vision application that demonstrates convolutional neural networks directly, compares a compact CNN, a deeper regularised CNN and ResNet18 transfer learning, then turns calibrated confidence into an auto-classify vs human-review decision.


## Data provenance
CIFAR-10 is loaded from `torchvision.datasets.CIFAR10`. Raw images are downloaded at runtime rather than committed. The benchmark contains 60,000 colour images in 10 classes.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(device)


## Image preprocessing, augmentation and validation split


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
])
train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=train_transform)
train_eval = datasets.CIFAR10(root='data', train=True, download=False, transform=eval_transform)
test_set = datasets.CIFAR10(root='data', train=False, download=True, transform=eval_transform)
indices = np.arange(len(train_full))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)
val_idx = indices[:5000]
train_idx = indices[5000:]
train_set = Subset(train_full, train_idx.tolist())
val_set = Subset(train_eval, val_idx.tolist())
print(len(train_set), len(val_set), len(test_set))


## Exploratory image analysis and class balance


In [ ]:
targets = np.array(train_full.targets)
class_counts = pd.Series(targets).value_counts().sort_index()
class_balance = pd.DataFrame({'class': class_names, 'images': class_counts.values})
display(class_balance)
plt.figure(figsize=(10,4))
plt.bar(class_balance['class'], class_balance['images'])
plt.xticks(rotation=45)
plt.ylabel('images')
plt.title('CIFAR-10 training class balance')
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(2,5,figsize=(12,5))
shown = set()
for image, label in train_full:
    if label in shown:
        continue
    tensor = image.clone()
    means = torch.tensor((0.4914,0.4822,0.4465)).view(3,1,1)
    stds = torch.tensor((0.2470,0.2435,0.2616)).view(3,1,1)
    tensor = (tensor * stds + means).clamp(0,1)
    ax = axes.flat[label]
    ax.imshow(tensor.permute(1,2,0))
    ax.set_title(class_names[label])
    ax.axis('off')
    shown.add(label)
    if len(shown) == 10:
        break
plt.tight_layout()
plt.show()


## Model 1 — compact CNN baseline


In [ ]:
compact_cnn = nn.Sequential(
    nn.Conv2d(3,32,3,padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32,64,3,padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(64,128,3,padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1,1)),
    nn.Flatten(),
    nn.Linear(128,10),
).to(device)
print(compact_cnn)
print('parameters:', sum(p.numel() for p in compact_cnn.parameters()))


## Model 2 — deeper regularised CNN with BatchNorm and Dropout


In [ ]:
regularized_cnn = nn.Sequential(
    nn.Conv2d(3,64,3,padding=1,bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.Conv2d(64,64,3,padding=1,bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Dropout(0.10),
    nn.Conv2d(64,128,3,padding=1,bias=False),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128,128,3,padding=1,bias=False),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Dropout(0.15),
    nn.Conv2d(128,256,3,padding=1,bias=False),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1,1)),
    nn.Flatten(),
    nn.Dropout(0.30),
    nn.Linear(256,10),
).to(device)
print(regularized_cnn)
print('parameters:', sum(p.numel() for p in regularized_cnn.parameters()))


## Model 3 — transfer learning with ResNet18


In [ ]:
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet18.conv1 = nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False)
resnet18.maxpool = nn.Identity()
resnet18.fc = nn.Linear(resnet18.fc.in_features,10)
resnet18 = resnet18.to(device)
print('ResNet18 parameters:', sum(p.numel() for p in resnet18.parameters()))


## Training configuration


In [ ]:
train_loader = DataLoader(train_set,batch_size=128,shuffle=True,num_workers=2)
val_loader = DataLoader(val_set,batch_size=128,shuffle=False,num_workers=2)
test_loader = DataLoader(test_set,batch_size=128,shuffle=False,num_workers=2)
model = regularized_cnn
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=8)
history = []
for epoch in range(1,9):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_seen = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_correct += (logits.argmax(1)==labels).sum().item()
        train_seen += images.size(0)
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_seen = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            val_correct += (logits.argmax(1)==labels).sum().item()
            val_seen += images.size(0)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss/train_seen,
        'train_accuracy': train_correct/train_seen,
        'val_loss': val_loss/val_seen,
        'val_accuracy': val_correct/val_seen,
    })
    scheduler.step()
history_df = pd.DataFrame(history)
display(history_df)


## Learning curves and overfitting check


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(history_df['epoch'],history_df['train_accuracy'],marker='o',label='train accuracy')
ax.plot(history_df['epoch'],history_df['val_accuracy'],marker='o',label='validation accuracy')
ax.set_xlabel('epoch')
ax.set_ylabel('accuracy')
ax.legend()
ax.set_title('CNN learning curve')
plt.show()
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(history_df['epoch'],history_df['train_loss'],marker='o',label='train loss')
ax.plot(history_df['epoch'],history_df['val_loss'],marker='o',label='validation loss')
ax.set_xlabel('epoch')
ax.set_ylabel('cross-entropy loss')
ax.legend()
ax.set_title('CNN loss curve')
plt.show()


## Evaluation, confusion matrix and per-class error analysis


In [ ]:
model.eval()
all_logits = []
all_targets = []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device)).cpu()
        all_logits.append(logits)
        all_targets.append(labels)
test_logits = torch.cat(all_logits)
test_targets = torch.cat(all_targets).numpy()
test_probabilities = torch.softmax(test_logits,dim=1).numpy()
test_predictions = test_probabilities.argmax(axis=1)
print('test accuracy:', accuracy_score(test_targets,test_predictions))
print(classification_report(test_targets,test_predictions,target_names=class_names))
cm = confusion_matrix(test_targets,test_predictions)
plt.figure(figsize=(8,8))
plt.imshow(cm)
plt.xticks(range(10),class_names,rotation=45,ha='right')
plt.yticks(range(10),class_names)
plt.xlabel('predicted')
plt.ylabel('actual')
plt.title('CNN confusion matrix')
plt.colorbar()
plt.tight_layout()
plt.show()
support = cm.sum(axis=1)
correct = np.diag(cm)
class_results = pd.DataFrame({
    'class': class_names,
    'support': support,
    'class_accuracy': correct/np.maximum(support,1),
})
class_results['error_rate'] = 1-class_results['class_accuracy']
display(class_results.sort_values('error_rate',ascending=False))


## Calibration and confidence-routing decision


In [ ]:
confidence = test_probabilities.max(axis=1)
correct = test_predictions == test_targets
routing_rows = []
for threshold in np.arange(0.50,0.96,0.05):
    accepted = confidence >= threshold
    routing_rows.append({
        'threshold': round(float(threshold),2),
        'coverage': float(accepted.mean()),
        'review_rate': float(1-accepted.mean()),
        'accepted_accuracy': float(correct[accepted].mean()) if accepted.any() else np.nan,
    })
routing = pd.DataFrame(routing_rows)
display(routing)
plt.figure(figsize=(8,4))
plt.plot(routing['threshold'],routing['coverage'],marker='o',label='automation coverage')
plt.plot(routing['threshold'],routing['accepted_accuracy'],marker='o',label='accepted accuracy')
plt.xlabel('confidence threshold')
plt.ylabel('rate')
plt.title('Confidence threshold trade-off')
plt.legend()
plt.show()


## Decision / application
The final model should be chosen from measured validation performance, calibration and compute cost rather than architecture size. In an operational image-routing system, high-confidence predictions can be automated and lower-confidence images should be routed to review.

## Explainability
For a production extension, Grad-CAM or activation-map inspection should be used on convolutional feature maps to verify that the network is responding to the image object rather than spurious background patterns.

## Reproducibility
Run `pip install -r requirements.txt` and then `python run.py --model regularized --epochs 8`. The Python implementation writes training, prediction, calibration and routing evidence to `artifacts/`.

## Limitations / next steps
CIFAR-10 is a benchmark, not a retailer-specific production dataset. A real deployment needs domain images, shift monitoring, licence/privacy review, latency tests, explicit error costs and a documented human-review policy.
